# Online Retail

- InvoiceNo	ID	Categorical	a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation
- StockCode	ID	Categorical	a 5-digit integral number uniquely assigned to each distinct product
- Description	Feature	Categorical	product name
- Quantity	Feature	Integer	the quantities of each product - (item) per transaction
- InvoiceDate	Feature	Date	the day and time when each - transaction was generated
- UnitPrice	Feature	Continuous	product price per unit	sterling
- CustomerID	Feature	Categorical	a 5-digit integral number - uniquely assigned to each customer
- Country	Feature	Categorical	the name of the country where each customer resides

In [222]:
import numpy as np
import pandas as pd 

# Ignorar avisos irrelevantes
import warnings
warnings.filterwarnings("ignore")

In [245]:
df = pd.read_excel("OnlineRetail.xlsx")
df = df.sample(n=50000, random_state=42)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
209268,555200,71459,HANGING JAM JAR T-LIGHT HOLDER,24,2011-06-01 12:05:00,0.85,17315.0,United Kingdom
207108,554974,21128,GOLD FISHING GNOME,4,2011-05-27 17:14:00,6.95,14031.0,United Kingdom
167085,550972,21086,SET/6 RED SPOTTY PAPER CUPS,4,2011-04-21 17:05:00,0.65,14031.0,United Kingdom
471836,576652,22812,PACK 3 BOXES CHRISTMAS PANETTONE,3,2011-11-16 10:39:00,1.95,17198.0,United Kingdom
115865,546157,22180,RETROSPOT LAMP,2,2011-03-10 08:40:00,9.95,13502.0,United Kingdom


In [246]:
df.shape

(50000, 8)

In [247]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

Investigámos a existencia de valores nulos.

In [248]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 50000 entries, 209268 to 442368
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    50000 non-null  object        
 1   StockCode    50000 non-null  object        
 2   Description  49860 non-null  object        
 3   Quantity     50000 non-null  int64         
 4   InvoiceDate  50000 non-null  datetime64[ns]
 5   UnitPrice    50000 non-null  float64       
 6   CustomerID   37396 non-null  float64       
 7   Country      50000 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 3.4+ MB


InvoiceNo          0
StockCode          0
Description      140
Quantity           0
InvoiceDate        0
UnitPrice          0
CustomerID     12604
Country            0
dtype: int64

In [249]:
df = df.dropna(subset=['CustomerID'])
print(df.shape)

(37396, 8)


In [250]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 37396 entries, 209268 to 442368
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    37396 non-null  object        
 1   StockCode    37396 non-null  object        
 2   Description  37396 non-null  object        
 3   Quantity     37396 non-null  int64         
 4   InvoiceDate  37396 non-null  datetime64[ns]
 5   UnitPrice    37396 non-null  float64       
 6   CustomerID   37396 non-null  float64       
 7   Country      37396 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 2.6+ MB


InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

ver duplicados

In [251]:
df.duplicated().sum()

np.int64(52)

In [252]:
duplicados = df[df.duplicated(keep=False)]

duplicados_ordenados = duplicados.sort_values(by=['InvoiceNo', 'StockCode', 'Quantity'])

duplicados_ordenados.head(12)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
8765,537144,21882,SKULLS TAPE,1,2010-12-05 13:00:00,0.65,15880.0,United Kingdom
8705,537144,21882,SKULLS TAPE,1,2010-12-05 13:00:00,0.65,15880.0,United Kingdom
25383,538368,22195,LARGE HEART MEASURING SPOONS,1,2010-12-12 10:57:00,1.65,15503.0,United Kingdom
25385,538368,22195,LARGE HEART MEASURING SPOONS,1,2010-12-12 10:57:00,1.65,15503.0,United Kingdom
34061,539317,22443,GROW YOUR OWN HERBS SET OF 3,1,2010-12-16 19:04:00,7.95,17392.0,United Kingdom
34050,539317,22443,GROW YOUR OWN HERBS SET OF 3,1,2010-12-16 19:04:00,7.95,17392.0,United Kingdom
51582,540647,84947,ANTIQUE SILVER TEA GLASS ENGRAVED,6,2011-01-10 14:57:00,1.25,17406.0,United Kingdom
51581,540647,84947,ANTIQUE SILVER TEA GLASS ENGRAVED,6,2011-01-10 14:57:00,1.25,17406.0,United Kingdom
83449,543306,22539,MINI JIGSAW DOLLY GIRL,1,2011-02-07 11:56:00,0.42,16686.0,United Kingdom
83437,543306,22539,MINI JIGSAW DOLLY GIRL,1,2011-02-07 11:56:00,0.42,16686.0,United Kingdom


In [244]:
df = df.drop_duplicates()
print(df.shape)

(37178, 8)


Investigar se os stockcodes correspondiam sempre à mesma description

In [232]:
df.groupby('StockCode')['Description'].nunique()[lambda x: x > 1]

StockCode
21112     2
21175     2
21232     2
21243     2
21507     2
         ..
84997B    2
84997C    2
84997D    2
85184C    2
85185B    2
Name: Description, Length: 105, dtype: int64

Fomos ver individualmente o que estava a acontecer e notou-se um padrão. Mas que eram necessariamente os mesmos produtos.

In [261]:
df[df['StockCode'].astype(str) == '21112']['Description'].unique()

array(['SWISS ROLL TOWEL PINK  SPOTS', 'SWISS ROLL TOWEL, PINK  SPOTS'],
      dtype=object)

In [262]:
df[df['StockCode'].astype(str) == '21175']['Description'].unique()

array(['GIN + TONIC DIET METAL SIGN', 'GIN AND TONIC DIET METAL SIGN'],
      dtype=object)

In [263]:
df[df['StockCode'].astype(str) == '21232']['Description'].unique()

array(['STRAWBERRY CERAMIC TRINKET BOX', 'STRAWBERRY CERAMIC TRINKET POT'],
      dtype=object)

In [265]:
df[df['StockCode'].astype(str) == '85185B']['Description'].unique()

array(['PINK HORSE SOCK PUPPET KIT', 'PINK HORSE SOCK PUPPET'],
      dtype=object)

## Limpar as variáveis numéricas que estavam como objeto

Observar os casos de StockCode que não estão numéricos.

In [233]:
df[~df['StockCode'].astype(str).str.isnumeric()]['StockCode'].value_counts()


StockCode
85123A    183
85099B    157
POST      109
82494L     94
85099F     64
         ... 
85049B      1
90184C      1
90082D      1
72800D      1
84951B      1
Name: count, Length: 519, dtype: int64

Investigar um caso concreto

In [234]:
df[df['StockCode'].astype(str).str.startswith('90214')].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
527050,580691,90214C,"LETTER ""C"" BLING KEY RING",12,2011-12-05 15:48:00,0.29,13790.0,United Kingdom
17502,537765,90214K,"LETTER ""K"" BLING KEY RING",1,2010-12-08 12:08:00,1.25,14606.0,United Kingdom
540095,581467,90214C,"LETTER ""C"" BLING KEY RING",1,2011-12-08 19:24:00,0.29,13077.0,United Kingdom
538792,581414,90214R,"LETTER ""R"" BLING KEY RING",1,2011-12-08 14:39:00,0.29,14730.0,United Kingdom
527048,580691,90214A,"LETTER ""A"" BLING KEY RING",12,2011-12-05 15:48:00,0.29,13790.0,United Kingdom


Investigar outro caso

In [235]:
df[df['StockCode'].astype(str).str.startswith('85123')].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
195717,553739,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2011-05-19 09:23:00,2.95,14895.0,United Kingdom
44938,540247,85123A,WHITE HANGING HEART T-LIGHT HOLDER,3,2011-01-05 15:56:00,2.95,15464.0,United Kingdom
183297,552655,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2,2011-05-10 14:22:00,2.95,14587.0,United Kingdom
12896,537400,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-06 14:36:00,2.95,17191.0,United Kingdom
68006,541849,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1,2011-01-23 13:34:00,2.95,13230.0,United Kingdom


Eliminar o que estiver à frente dos 5 digitos

In [236]:
df['StockCode'] = df['StockCode'].astype(str).str[:5]

Ver se ainda há letras ou símbolos no stockcode.

In [237]:
df[~df['StockCode'].astype(str).str.isnumeric()]['StockCode'].value_counts()

StockCode
POST     109
M         33
C2        12
D          4
CRUK       3
BANK       2
PADS       1
DOT        1
Name: count, dtype: int64

Eliminar o que restava e passar a variável para numérica.

In [238]:
df = df[df['StockCode'].astype(str).str.isnumeric()]
df['StockCode'] = df['StockCode'].astype(int)

Ver se tudo parece bater certo.

In [239]:
df.sort_values('StockCode').head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
65899,541698,10002,INFLATABLE POLITICAL GLOBE,18,2011-01-20 19:16:00,0.85,14713.0,United Kingdom
127450,547223,10002,INFLATABLE POLITICAL GLOBE,5,2011-03-21 15:10:00,0.85,12867.0,United Kingdom
31,536370,10002,INFLATABLE POLITICAL GLOBE,48,2010-12-01 08:45:00,0.85,12583.0,France
142354,548606,10002,INFLATABLE POLITICAL GLOBE,120,2011-04-01 11:10:00,0.85,12731.0,France
20617,538069,10002,INFLATABLE POLITICAL GLOBE,8,2010-12-09 14:08:00,0.85,16795.0,United Kingdom


Ver o resultado final.

In [240]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37179 entries, 209268 to 442368
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    37179 non-null  object        
 1   StockCode    37179 non-null  int64         
 2   Description  37179 non-null  object        
 3   Quantity     37179 non-null  int64         
 4   InvoiceDate  37179 non-null  datetime64[ns]
 5   UnitPrice    37179 non-null  float64       
 6   CustomerID   37179 non-null  float64       
 7   Country      37179 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(3)
memory usage: 2.6+ MB


Viu-se que o InvoiceNo também ainda está como objeto e deve ser portando tratado.

Observar os casos de InvoiceNo que não estão numéricos.

In [241]:
df[~df['InvoiceNo'].astype(str).str.isnumeric()]['InvoiceNo'].value_counts()

InvoiceNo
C570867    9
C548460    8
C569985    6
C569655    6
C560540    5
          ..
C550027    1
C560932    1
C551465    1
C551551    1
C546730    1
Name: count, Length: 638, dtype: int64

Eliminar as compras que foram canceladas

In [242]:
df = df[df['StockCode'].astype(str).str.isnumeric()]
df['StockCode'] = df['StockCode'].astype(int)

Ver o resultado final.

In [243]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37179 entries, 209268 to 442368
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   InvoiceNo    37179 non-null  object        
 1   StockCode    37179 non-null  int64         
 2   Description  37179 non-null  object        
 3   Quantity     37179 non-null  int64         
 4   InvoiceDate  37179 non-null  datetime64[ns]
 5   UnitPrice    37179 non-null  float64       
 6   CustomerID   37179 non-null  float64       
 7   Country      37179 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(3)
memory usage: 2.6+ MB
